In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from lunardate import LunarDate
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
# Block 0 - Experiement Global Config
@dataclass(frozen=True)
class ExperimentConfig:
    # Data paths
    data_path: Path = Path("data/sales_daily.csv")

    # Split ranges, inclusive YYYYMM.
    train_ym_range: tuple[int, int] = (202201, 202412)
    valid_ym_range: tuple[int, int] = (202501, 202512)
    test_ym_range: tuple[int, int] = (202601, 202612)

    # Rolling daily forecast target. 
    # Max rolling daily horizon. 31 covers full next-month forecast from prior month end.
    forecast_horizon_days: int = 31

    # Evaluation anchor:
    # If eval_forecast_start_wd_seq=10, model uses information up to WD9 (9th working day)
    # and evaluates predicted daily qty from WD10 to month end.
    eval_forecast_start_wd_seq: int = 10

    # Plot one test month forecast path.
    display_bizym: int = 202601

    # Calendar behavior.
    use_chinese_calendar_if_available: bool = True
    fill_missing_daily_qty_with_zero: bool = True

    # Feature windows.
    rolling_windows: tuple[int, ...] = (3, 5, 7, 14, 30)
    lag_days: tuple[int, ...] = (1, 2, 3, 7, 14, 28)
    same_month_history_years: tuple[int, ...] = (1, 2, 3)

    # Model selection.
    random_seed: int = 42
    cv_splits: int = 3
    max_param_candidates_per_model: int = 12
    target_col: str = "target_daily_qty"

    # Metric for model selection.
    selection_metric: str = "valid_month_total_mape"

    # Numerical safety.
    eps: float = 1e-9


CONFIG = ExperimentConfig()

MODEL_SPECS: dict[str, dict[str, Any]] = {
    "ridge": {
        "estimator": TransformedTargetRegressor(
            regressor=Pipeline([
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
                ("model", Ridge(random_state=CONFIG.random_seed)),
            ]),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        "param_grid": {
            "regressor__model__alpha": [0.1, 1.0, 3.0, 10.0, 30.0],
        },
    },
    "elastic_net": {
        "estimator": TransformedTargetRegressor(
            regressor=Pipeline([
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
                ("model", ElasticNet(random_state=CONFIG.random_seed, max_iter=20000)),
            ]),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        "param_grid": {
            "regressor__model__alpha": [0.001, 0.01, 0.1],
            "regressor__model__l1_ratio": [0.2, 0.5, 0.8],
        },
    },
    "huber": {
        "estimator": TransformedTargetRegressor(
            regressor=Pipeline([
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
                ("model", HuberRegressor(max_iter=2000)),
            ]),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        "param_grid": {
            "regressor__model__epsilon": [1.2, 1.35, 1.5],
            "regressor__model__alpha": [0.0001, 0.001, 0.01],
        },
    },
    "random_forest": {
        "estimator": Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", RandomForestRegressor(random_state=CONFIG.random_seed, n_jobs=-1)),
        ]),
        "param_grid": {
            "model__n_estimators": [200, 500],
            "model__max_depth": [2, 3, 4, None],
            "model__min_samples_leaf": [1, 2, 4],
        },
    },
    "extra_trees": {
        "estimator": Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", ExtraTreesRegressor(random_state=CONFIG.random_seed, n_jobs=-1)),
        ]),
        "param_grid": {
            "model__n_estimators": [200, 500],
            "model__max_depth": [2, 3, 4, None],
            "model__min_samples_leaf": [1, 2, 4],
        },
    },
    "gradient_boosting": {
        "estimator": Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", GradientBoostingRegressor(random_state=CONFIG.random_seed)),
        ]),
        "param_grid": {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [1, 3],
        },
    },
    "svr": {
        "estimator": TransformedTargetRegressor(
            regressor=Pipeline([
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
                ("model", SVR()),
            ]),
            func=np.log1p,
            inverse_func=np.expm1,
        ),
        "param_grid": {
            "regressor__model__C": [0.3, 1.0, 3.0, 10.0],
            "regressor__model__epsilon": [0.01, 0.05, 0.1],
            "regressor__model__gamma": ["scale", "auto"],
        },
    },
}

np.random.seed(CONFIG.random_seed)

In [ ]:
# Block 1 - Data Loading and Empty Fill

def resolve_data_path(path: Path) -> Path:
    candidates = [
        path,
        Path("code/30d-jenny") / path,
        Path("../../code/30d-jenny") / path,
        Path("data/sales_daily.csv"),
        Path("../../data/sales_daily.csv"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Cannot find data file. Tried: {[str(p) for p in candidates]}")


def build_is_workday_func(use_chinese_calendar_if_available: bool):
    if use_chinese_calendar_if_available:
        try:
            from chinese_calendar import is_workday as cn_is_workday

            return lambda ts: bool(cn_is_workday(pd.Timestamp(ts).date())), "chinese_calendar"
        except Exception:
            pass

    return lambda ts: pd.Timestamp(ts).dayofweek < 5, "weekday_fallback_mon_fri"

def to_lunar_fields(ts) -> dict[str, int]:
    from lunardate import LunarDate

    ts = pd.Timestamp(ts)
    lunar = LunarDate.fromSolarDate(ts.year, ts.month, ts.day)

    return {
        "lunar_year": lunar.year,
        "lunar_month": lunar.month,
        "lunar_day": lunar.day,
        "lunar_date": f"{lunar.year:04d}-{lunar.month:02d}-{lunar.day:02d}",
        "is_lunar_leap_month": int(bool(lunar.isLeapMonth)),
    }

def month_start_from_bizym(bizym: int) -> pd.Timestamp:
    year, month = divmod(int(bizym), 100)
    return pd.Timestamp(year=year, month=month, day=1)

def get_lunar_holiday(ts) -> str:
    ts = pd.Timestamp(ts)
    lunar = LunarDate.fromSolarDate(ts.year, ts.month, ts.day)
    next_lunar = LunarDate.fromSolarDate(
        (ts + pd.Timedelta(days=1)).year,
        (ts + pd.Timedelta(days=1)).month,
        (ts + pd.Timedelta(days=1)).day,
    )

    if next_lunar.month == 1 and next_lunar.day == 1:
        return "除夕"

    holiday_map = {
        (1, 1): "春节",
        (1, 2): "春节",
        (1, 3): "春节",
        (1, 4): "春节",
        (1, 5): "春节",
        (1, 6): "春节",
        (1, 7): "春节",
        (1, 15): "元宵",
        (5, 5): "端午",
        (7, 7): "七夕",
        (8, 15): "中秋",
        (9, 9): "重阳",
        (12, 8): "腊八",
    }

    return holiday_map.get((lunar.month, lunar.day), "")

def build_month_calendar(bizym: int, is_workday_func) -> pd.DataFrame:
    month_start = month_start_from_bizym(bizym)
    dates = pd.date_range(month_start, month_start + pd.offsets.MonthEnd(0), freq="D")

    out = pd.DataFrame({"transdate": dates})
    out["bizym"] = int(bizym)
    out["year"] = out["transdate"].dt.year
    out["month_num"] = out["transdate"].dt.month
    out["day_of_month"] = out["transdate"].dt.day
    out["day_of_week"] = out["transdate"].dt.dayofweek
    out["is_month_start"] = out["day_of_month"].eq(1).astype(int)
    out["is_month_end"] = out["transdate"].dt.is_month_end.astype(int)
    out["days_in_month"] = out["transdate"].dt.days_in_month
    out["days_to_month_end"] = out["days_in_month"] - out["day_of_month"]
    out["is_workday"] = out["transdate"].map(is_workday_func)
    out["workday_seq"] = out["is_workday"].cumsum().where(out["is_workday"], 0).astype(int)
    out["max_workday_seq"] = int(out["workday_seq"].max())
    out["workdays_to_month_end"] = np.where(
        out["is_workday"],
        out["max_workday_seq"] - out["workday_seq"],
        np.nan,
    )
    lunar_fields = out["transdate"].map(to_lunar_fields).apply(pd.Series)
    out = pd.concat([out, lunar_fields], axis=1)
    out["lunar_holiday"] = out["transdate"].map(get_lunar_holiday)
    out["is_lunar_new_year"] = out["lunar_holiday"].eq("春节").astype(int)
    out["is_lunar_new_year_eve"] = out["lunar_holiday"].eq("除夕").astype(int)
    out["is_spring_festival_core"] = out["lunar_holiday"].isin([
        "除夕",
        "春节",
        "端午",
        "中秋"
    ]).astype(int)
    out["is_lunar_holiday"] = out["lunar_holiday"].ne("").astype(int)
    return out


def load_daily_panel(config: ExperimentConfig) -> tuple[pd.DataFrame, Path, str]:
    data_path = resolve_data_path(config.data_path)
    is_workday_func, calendar_source = build_is_workday_func(config.use_chinese_calendar_if_available)

    raw = pd.read_csv(data_path)
    required_columns = {"bizym", "transdate", "qty", "num_hosp"}
    missing_columns = sorted(required_columns - set(raw.columns))
    if missing_columns:
        raise ValueError(f"{data_path} missing required columns: {missing_columns}")

    raw = raw[["bizym", "transdate", "qty", "num_hosp"]].copy()
    raw["bizym"] = raw["bizym"].astype(int)
    raw["transdate"] = pd.to_datetime(raw["transdate"], errors="coerce")
    raw["qty"] = pd.to_numeric(raw["qty"], errors="coerce")
    raw["num_hosp"] = pd.to_numeric(raw["num_hosp"], errors="coerce")

    # Negative qty means returns or accounting adjustments, not sellable demand.
    # Clip it before building labels so downstream log-target models stay valid.
    raw["qty"] = raw["qty"].clip(lower=0)
    raw["num_hosp"] = raw["num_hosp"].clip(lower=0)

    if raw["transdate"].isna().any():
        raise ValueError("transdate contains invalid values")

    monthly_calendars = [
        build_month_calendar(bizym, is_workday_func)
        for bizym in sorted(raw["bizym"].unique())
    ]
    calendar_df = pd.concat(monthly_calendars, ignore_index=True)

    daily = calendar_df.merge(
        raw,
        on=["bizym", "transdate"],
        how="left",
    )

    if config.fill_missing_daily_qty_with_zero:
        daily[["qty", "num_hosp"]] = daily[["qty", "num_hosp"]].fillna(0.0)

    daily = daily.sort_values(["bizym", "transdate"]).reset_index(drop=True)

    daily["mtd_qty"] = daily.groupby("bizym")["qty"].cumsum()
    daily["mtd_num_hosp"] = daily.groupby("bizym")["num_hosp"].cumsum()
    daily["month_total_qty"] = daily.groupby("bizym")["qty"].transform("sum")
    daily["month_total_num_hosp"] = daily.groupby("bizym")["num_hosp"].transform("sum")

    daily["mtd_qty_pct"] = daily["mtd_qty"] / daily["month_total_qty"].replace(0, np.nan)
    daily["mtd_num_hosp_pct"] = daily["mtd_num_hosp"] / daily["month_total_num_hosp"].replace(0, np.nan)

    return daily, data_path, calendar_source


daily_df, resolved_data_path, calendar_source = load_daily_panel(CONFIG)
front_cols = ["transdate", "lunar_date", "bizym", "year", "month_num", "day_of_month", "day_of_week", "lunar_year", "lunar_month", "lunar_day", "lunar_holiday", "is_lunar_leap_month", "is_lunar_holiday", "is_lunar_new_year", "is_lunar_new_year_eve", "is_spring_festival_core"]
daily_df = daily_df.reindex(columns=front_cols + [c for c in daily_df.columns if c not in front_cols])

print(f"Data path: {resolved_data_path.resolve()}")
print(f"Calendar source: {calendar_source}")
print(f"daily_df shape: {daily_df.shape}")
display(daily_df.loc[daily_df["lunar_holiday"].ne("")].head())

In [ ]:
# Block2 - Build Supervised Rolling Forecast Panel

# split train / valid / test based on the configured bizym range 
def assign_split_by_bizym(bizym: int, config: ExperimentConfig) -> str:
    if config.train_ym_range[0] <= bizym <= config.train_ym_range[1]:
        return "train"
    if config.valid_ym_range[0] <= bizym <= config.valid_ym_range[1]:
        return "valid"
    if config.test_ym_range[0] <= bizym <= config.test_ym_range[1]:
        return "test"
    return "unused"


# Build a supervised learning panel by aligning each anchor date with future target dates
def build_supervised_panel(daily: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    base = daily.sort_values(["bizym", "transdate"]).copy()

    future_parts = []

    for horizon in range(1, config.forecast_horizon_days + 1):
        future = base[["bizym", "transdate", "qty"]].copy()
        future["anchor_date"] = future["transdate"] - pd.to_timedelta(horizon, unit="D")
        future = future.rename(
            columns={
                "transdate": "target_date",
                "qty": config.target_col,
            }
        )
        future["horizon_days"] = horizon
        future_parts.append(future)

    future_panel = pd.concat(future_parts, ignore_index=True)

    anchor_cols = [
        "bizym",
        "transdate",
        "year",
        "month_num",
        "day_of_month",
        "day_of_week",
        "is_month_start",
        "is_month_end",
        "days_in_month",
        "days_to_month_end",
        "is_workday",
        "workday_seq",
        "max_workday_seq",
        "workdays_to_month_end",
        "qty",
        "num_hosp",
        "mtd_qty",
        "mtd_num_hosp",
        "month_total_qty",
        "month_total_num_hosp",
        "mtd_qty_pct",
        "mtd_num_hosp_pct",
    ]

    anchor_panel = base[anchor_cols].rename(
        columns={
            "transdate": "anchor_date",
            "qty": "anchor_day_qty",
            "num_hosp": "anchor_day_num_hosp",
        }
    )

    supervised = future_panel.merge(
        anchor_panel,
        on=["bizym", "anchor_date"],
        how="inner",
    )

    supervised["target_day_of_month"] = supervised["target_date"].dt.day
    supervised["target_day_of_week"] = supervised["target_date"].dt.dayofweek
    supervised["target_is_month_start"] = supervised["target_day_of_month"].eq(1).astype(int)
    supervised["target_is_month_end"] = supervised["target_date"].dt.is_month_end.astype(int)
    supervised["target_days_to_month_end"] = supervised["target_date"].dt.days_in_month - supervised["target_day_of_month"]

    supervised["split"] = supervised["bizym"].map(lambda ym: assign_split_by_bizym(int(ym), config))

    supervised = supervised[supervised["split"] != "unused"].copy()
    supervised = supervised.sort_values(["anchor_date", "target_date"]).reset_index(drop=True)

    return supervised


supervised_df = build_supervised_panel(daily_df, CONFIG)

print(f"supervised_df shape: {supervised_df.shape}")
display(supervised_df.head())
display(supervised_df.tail())
display(supervised_df["split"].value_counts().rename("rows").reset_index())

In [ ]:
# Block3 - Feature Engineering

def add_lag_features(df: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    out = df.sort_values(["bizym", "transdate"]).copy()

    for lag in config.lag_days:
        out[f"qty_lag_{lag}d"] = out.groupby("bizym")["qty"].shift(lag)
        out[f"num_hosp_lag_{lag}d"] = out.groupby("bizym")["num_hosp"].shift(lag)

    return out


def add_rolling_features(df: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    out = df.sort_values(["bizym", "transdate"]).copy()

    for window in config.rolling_windows:
        out[f"qty_roll_{window}d_mean"] = out.groupby("bizym")["qty"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )
        out[f"qty_roll_{window}d_std"] = out.groupby("bizym")["qty"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=2).std()
        )
        out[f"qty_roll_{window}d_sum"] = out.groupby("bizym")["qty"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).sum()
        )
        out[f"num_hosp_roll_{window}d_mean"] = out.groupby("bizym")["num_hosp"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )
        out[f"num_hosp_roll_{window}d_std"] = out.groupby("bizym")["num_hosp"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=2).std()
        )

    return out



def lunar_new_year_date(lunar_year: int) -> pd.Timestamp:
    return pd.Timestamp(LunarDate(int(lunar_year), 1, 1).toSolarDate())


def add_alignment_calendar_features(daily: pd.DataFrame) -> pd.DataFrame:
    out = daily.sort_values(["bizym", "transdate"]).copy()

    out["weekday_occurrence"] = out.groupby(["bizym", "day_of_week"]).cumcount() + 1
    out["weekday_occurrence_key"] = out["weekday_occurrence"] * 10 + out["day_of_week"]
    out["workday_progress"] = out["workday_seq"] / out["max_workday_seq"].replace(0, np.nan)
    out["daily_qty_share"] = out["qty"] / out["month_total_qty"].replace(0, np.nan)
    out["daily_num_hosp_share"] = out["num_hosp"] / out["month_total_num_hosp"].replace(0, np.nan)
    out["mtd_qty_share"] = out["mtd_qty"] / out["month_total_qty"].replace(0, np.nan)
    out["mtd_num_hosp_share"] = out["mtd_num_hosp"] / out["month_total_num_hosp"].replace(0, np.nan)

    prev_lny = out["lunar_year"].map(lunar_new_year_date)
    next_lny = (out["lunar_year"] + 1).map(lunar_new_year_date)
    days_since_prev_lny = (out["transdate"] - prev_lny).dt.days
    days_to_next_lny = (next_lny - out["transdate"]).dt.days

    out["days_since_lunar_new_year"] = days_since_prev_lny
    out["days_to_next_lunar_new_year"] = days_to_next_lny
    out["days_from_nearest_lunar_new_year"] = np.where(
        days_since_prev_lny <= days_to_next_lny,
        days_since_prev_lny,
        -days_to_next_lny,
    )
    out["abs_days_from_nearest_lunar_new_year"] = out["days_from_nearest_lunar_new_year"].abs()
    out["is_lunar_new_year_window_7d"] = out["abs_days_from_nearest_lunar_new_year"].le(7).astype(int)
    out["is_lunar_new_year_window_14d"] = out["abs_days_from_nearest_lunar_new_year"].le(14).astype(int)
    out["is_lunar_new_year_window_30d"] = out["abs_days_from_nearest_lunar_new_year"].le(30).astype(int)

    return out

def add_daily_history_features(daily: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    out = add_alignment_calendar_features(daily)
    out["qty_per_hosp"] = out["qty"] / out["num_hosp"].replace(0, np.nan)

    out = add_lag_features(out, config)
    out = add_rolling_features(out, config)

    for window in config.rolling_windows:
        out[f"qty_per_hosp_roll_{window}d_mean"] = out.groupby("bizym")["qty_per_hosp"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=1).mean()
        )
        out[f"qty_per_hosp_roll_{window}d_std"] = out.groupby("bizym")["qty_per_hosp"].transform(
            lambda s: s.shift(1).rolling(window, min_periods=2).std()
        )

    history_feature_cols = [
        "bizym",
        "transdate",
        "qty_per_hosp",
        "weekday_occurrence",
        "weekday_occurrence_key",
        "workday_progress",
        "days_since_lunar_new_year",
        "days_to_next_lunar_new_year",
        "days_from_nearest_lunar_new_year",
        "abs_days_from_nearest_lunar_new_year",
        "is_lunar_new_year_window_7d",
        "is_lunar_new_year_window_14d",
        "is_lunar_new_year_window_30d",
    ]

    for lag in config.lag_days:
        history_feature_cols.append(f"qty_lag_{lag}d")
        history_feature_cols.append(f"num_hosp_lag_{lag}d")

    for window in config.rolling_windows:
        history_feature_cols.append(f"qty_roll_{window}d_mean")
        history_feature_cols.append(f"qty_roll_{window}d_std")
        history_feature_cols.append(f"qty_roll_{window}d_sum")
        history_feature_cols.append(f"num_hosp_roll_{window}d_mean")
        history_feature_cols.append(f"num_hosp_roll_{window}d_std")
        history_feature_cols.append(f"qty_per_hosp_roll_{window}d_mean")
        history_feature_cols.append(f"qty_per_hosp_roll_{window}d_std")

    return out[history_feature_cols].rename(
        columns={
            "transdate": "anchor_date",
            "weekday_occurrence": "anchor_weekday_occurrence",
            "weekday_occurrence_key": "anchor_weekday_occurrence_key",
            "workday_progress": "anchor_workday_progress_calendar",
            "days_since_lunar_new_year": "anchor_days_since_lunar_new_year",
            "days_to_next_lunar_new_year": "anchor_days_to_next_lunar_new_year",
            "days_from_nearest_lunar_new_year": "anchor_days_from_nearest_lunar_new_year",
            "abs_days_from_nearest_lunar_new_year": "anchor_abs_days_from_nearest_lunar_new_year",
            "is_lunar_new_year_window_7d": "anchor_is_lunar_new_year_window_7d",
            "is_lunar_new_year_window_14d": "anchor_is_lunar_new_year_window_14d",
            "is_lunar_new_year_window_30d": "anchor_is_lunar_new_year_window_30d",
        }
    )


def add_same_month_history_features(panel: pd.DataFrame, daily: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    out = panel.copy()

    monthly = (
        daily.groupby("bizym", as_index=False)
        .agg(
            hist_month_total_qty=("qty", "sum"),
            hist_month_total_num_hosp=("num_hosp", "sum"),
            hist_month_workdays=("is_workday", "sum"),
            hist_month_days=("transdate", "nunique"),
        )
    )

    for years_back in config.same_month_history_years:
        lag_monthly = monthly.copy()
        lag_monthly["bizym"] = lag_monthly["bizym"] + years_back * 100
        lag_monthly = lag_monthly.rename(
            columns={
                "hist_month_total_qty": f"same_month_y{years_back}_total_qty",
                "hist_month_total_num_hosp": f"same_month_y{years_back}_total_num_hosp",
                "hist_month_workdays": f"same_month_y{years_back}_workdays",
                "hist_month_days": f"same_month_y{years_back}_days",
            }
        )
        out = out.merge(lag_monthly, on="bizym", how="left")

    same_month_qty_cols = [
        f"same_month_y{years_back}_total_qty"
        for years_back in config.same_month_history_years
    ]
    same_month_hosp_cols = [
        f"same_month_y{years_back}_total_num_hosp"
        for years_back in config.same_month_history_years
    ]

    out["same_month_total_qty_mean"] = out[same_month_qty_cols].mean(axis=1)
    out["same_month_total_qty_std"] = out[same_month_qty_cols].std(axis=1)
    out["same_month_total_num_hosp_mean"] = out[same_month_hosp_cols].mean(axis=1)
    out["same_month_total_num_hosp_std"] = out[same_month_hosp_cols].std(axis=1)

    return out



def add_alignment_history_features(panel: pd.DataFrame, daily: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    out = panel.copy()
    hist = add_alignment_calendar_features(daily)
    share_metrics = [
        "daily_qty_share",
        "mtd_qty_share",
        "daily_num_hosp_share",
        "mtd_num_hosp_share",
    ]

    feature_groups: dict[str, list[str]] = {}

    def merge_profile(
        base_profile: pd.DataFrame,
        merge_keys: list[str],
        rename_keys: dict[str, str],
        prefix: str,
        years_back: int,
    ) -> None:
        nonlocal out
        profile = base_profile[list(rename_keys.keys()) + share_metrics].copy()
        profile = profile.rename(columns=rename_keys)
        rename_metrics = {
            metric: f"same_y{years_back}_{prefix}_{metric}"
            for metric in share_metrics
        }
        profile = profile.rename(columns=rename_metrics)
        out = out.merge(profile, on=merge_keys, how="left")
        for metric, feature_col in rename_metrics.items():
            feature_groups.setdefault(f"{prefix}_{metric}", []).append(feature_col)

    for years_back in config.same_month_history_years:
        shifted = hist.copy()
        shifted["bizym"] = shifted["bizym"] + years_back * 100

        merge_profile(
            shifted,
            merge_keys=["bizym", "target_day_of_month"],
            rename_keys={"bizym": "bizym", "day_of_month": "target_day_of_month"},
            prefix="calendar_day",
            years_back=years_back,
        )

        merge_profile(
            shifted.query("workday_seq > 0"),
            merge_keys=["bizym", "target_workday_seq"],
            rename_keys={"bizym": "bizym", "workday_seq": "target_workday_seq"},
            prefix="workday_seq",
            years_back=years_back,
        )

        merge_profile(
            shifted,
            merge_keys=["bizym", "target_day_of_week", "target_weekday_occurrence"],
            rename_keys={
                "bizym": "bizym",
                "day_of_week": "target_day_of_week",
                "weekday_occurrence": "target_weekday_occurrence",
            },
            prefix="weekday_occurrence",
            years_back=years_back,
        )

        merge_profile(
            shifted.query("workday_seq > 0"),
            merge_keys=["bizym", "workday_seq"],
            rename_keys={"bizym": "bizym", "workday_seq": "workday_seq"},
            prefix="anchor_workday_seq",
            years_back=years_back,
        )

        lunar_shifted = hist.copy()
        lunar_shifted["target_lunar_year"] = lunar_shifted["lunar_year"] + years_back
        merge_profile(
            lunar_shifted,
            merge_keys=[
                "target_lunar_year",
                "target_lunar_month",
                "target_lunar_day",
                "target_is_lunar_leap_month",
            ],
            rename_keys={
                "target_lunar_year": "target_lunar_year",
                "lunar_month": "target_lunar_month",
                "lunar_day": "target_lunar_day",
                "is_lunar_leap_month": "target_is_lunar_leap_month",
            },
            prefix="lunar_day",
            years_back=years_back,
        )

    for group_name, cols in feature_groups.items():
        out[f"hist_{group_name}_mean"] = out[cols].mean(axis=1)
        out[f"hist_{group_name}_std"] = out[cols].std(axis=1)

    out["expected_anchor_mtd_qty_by_workday_seq"] = (
        out["same_month_total_qty_mean"] * out["hist_anchor_workday_seq_mtd_qty_share_mean"]
    )
    out["anchor_mtd_vs_expected_workday_seq"] = (
        out["mtd_qty"] / out["expected_anchor_mtd_qty_by_workday_seq"].replace(0, np.nan)
    )
    out["expected_target_daily_qty_by_workday_seq"] = (
        out["same_month_total_qty_mean"] * out["hist_workday_seq_daily_qty_share_mean"]
    )
    out["expected_target_daily_qty_by_weekday_occurrence"] = (
        out["same_month_total_qty_mean"] * out["hist_weekday_occurrence_daily_qty_share_mean"]
    )
    out["expected_target_daily_qty_by_lunar_day"] = (
        out["same_month_total_qty_mean"] * out["hist_lunar_day_daily_qty_share_mean"]
    )

    return out

def build_feature_panel(supervised: pd.DataFrame, daily: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    out = supervised.copy()
    daily = add_alignment_calendar_features(daily)

    target_calendar_cols = [
        "bizym",
        "transdate",
        "workday_seq",
        "max_workday_seq",
        "workdays_to_month_end",
        "weekday_occurrence",
        "weekday_occurrence_key",
        "lunar_year",
        "lunar_month",
        "lunar_day",
        "is_lunar_leap_month",
        "days_since_lunar_new_year",
        "days_to_next_lunar_new_year",
        "days_from_nearest_lunar_new_year",
        "abs_days_from_nearest_lunar_new_year",
        "is_lunar_new_year_window_7d",
        "is_lunar_new_year_window_14d",
        "is_lunar_new_year_window_30d",
        "is_lunar_new_year",
        "is_lunar_new_year_eve",
        "is_spring_festival_core",
        "is_lunar_holiday",
    ]

    target_calendar = daily[target_calendar_cols].rename(
        columns={
            "transdate": "target_date",
            "workday_seq": "target_workday_seq",
            "max_workday_seq": "target_max_workday_seq",
            "workdays_to_month_end": "target_workdays_to_month_end",
            "weekday_occurrence": "target_weekday_occurrence",
            "weekday_occurrence_key": "target_weekday_occurrence_key",
            "lunar_year": "target_lunar_year",
            "lunar_month": "target_lunar_month",
            "lunar_day": "target_lunar_day",
            "is_lunar_leap_month": "target_is_lunar_leap_month",
            "days_since_lunar_new_year": "target_days_since_lunar_new_year",
            "days_to_next_lunar_new_year": "target_days_to_next_lunar_new_year",
            "days_from_nearest_lunar_new_year": "target_days_from_nearest_lunar_new_year",
            "abs_days_from_nearest_lunar_new_year": "target_abs_days_from_nearest_lunar_new_year",
            "is_lunar_new_year_window_7d": "target_is_lunar_new_year_window_7d",
            "is_lunar_new_year_window_14d": "target_is_lunar_new_year_window_14d",
            "is_lunar_new_year_window_30d": "target_is_lunar_new_year_window_30d",
            "is_lunar_new_year": "target_is_lunar_new_year",
            "is_lunar_new_year_eve": "target_is_lunar_new_year_eve",
            "is_spring_festival_core": "target_is_spring_festival_core",
            "is_lunar_holiday": "target_is_lunar_holiday",
        }
    )

    out = out.merge(target_calendar, on=["bizym", "target_date"], how="left")    

    daily_history_features = add_daily_history_features(daily, config)
    out = out.merge(daily_history_features, on=["bizym", "anchor_date"], how="left")

    out = add_same_month_history_features(out, daily, config)
    out = add_alignment_history_features(out, daily, config)

    out["anchor_month_progress"] = out["day_of_month"] / out["days_in_month"].replace(0, np.nan)
    out["target_month_progress"] = out["target_day_of_month"] / out["days_in_month"].replace(0, np.nan)
    out["anchor_workday_progress"] = out["workday_seq"] / out["max_workday_seq"].replace(0, np.nan)
    out["target_workday_progress"] = out["target_workday_seq"] / out["target_max_workday_seq"].replace(0, np.nan)
    out["target_is_after_anchor"] = (out["target_date"] > out["anchor_date"]).astype(int)
    out["anchor_mtd_qty_per_workday"] = out["mtd_qty"] / out["workday_seq"].replace(0, np.nan)
    out["anchor_mtd_num_hosp_per_workday"] = out["mtd_num_hosp"] / out["workday_seq"].replace(0, np.nan)
    out["anchor_mtd_qty_per_hosp"] = out["mtd_qty"] / out["mtd_num_hosp"].replace(0, np.nan)
    out["anchor_day_qty_per_hosp"] = out["anchor_day_qty"] / out["anchor_day_num_hosp"].replace(0, np.nan)
    out["anchor_mtd_vs_same_month_mean"] = out["mtd_qty"] / out["same_month_total_qty_mean"].replace(0, np.nan)
    out["anchor_mtd_hosp_vs_same_month_mean"] = out["mtd_num_hosp"] / out["same_month_total_num_hosp_mean"].replace(0, np.nan)
    out["same_month_qty_cv"] = out["same_month_total_qty_std"] / out["same_month_total_qty_mean"].replace(0, np.nan)
    out["same_month_num_hosp_cv"] = out["same_month_total_num_hosp_std"] / out["same_month_total_num_hosp_mean"].replace(0, np.nan)

    out["is_target_workday"] = out["target_day_of_week"].isin([0, 1, 2, 3, 4]).astype(int)
    out["is_target_weekend"] = out["target_day_of_week"].isin([5, 6]).astype(int)
    out["target_week_of_month"] = ((out["target_day_of_month"] - 1) // 7 + 1).astype(int)
    out["anchor_week_of_month"] = ((out["day_of_month"] - 1) // 7 + 1).astype(int)
    out["target_is_jan_feb"] = out["month_num"].isin([1, 2]).astype(int)
    out["target_is_same_lunar_month_as_spring_festival"] = out["target_lunar_month"].eq(1).astype(int)

    # out["target_is_lunar_new_year"] = out["is_lunar_new_year"]
    # out["target_is_lunar_new_year_eve"] = out["is_lunar_new_year_eve"]
    # out["target_is_spring_festival_core"] = out["is_spring_festival_core"]
    # out["target_is_lunar_holiday"] = out["is_lunar_holiday"]

    out = out.replace([np.inf, -np.inf], np.nan)

    return out


all_fe = build_feature_panel(supervised_df, daily_df, CONFIG)

LEAKAGE_COLUMNS = {
    CONFIG.target_col,
    "month_total_qty",
    "month_total_num_hosp",
}

IDENTIFIER_COLUMNS = {
    "anchor_date",
    "target_date",
    "split",
    "lunar_date",
    "lunar_holiday",
}

FEATURE_COLUMNS = [
    col
    for col in all_fe.columns
    if col not in LEAKAGE_COLUMNS
    and col not in IDENTIFIER_COLUMNS
    and pd.api.types.is_numeric_dtype(all_fe[col])
]

train_fe = all_fe[all_fe["split"].eq("train")].copy()
valid_fe = all_fe[all_fe["split"].eq("valid")].copy()
test_fe = all_fe[all_fe["split"].eq("test")].copy()

print(f"all_fe shape: {all_fe.shape}")
print(f"train_fe shape: {train_fe.shape}")
print(f"valid_fe shape: {valid_fe.shape}")
print(f"test_fe shape: {test_fe.shape}")
print(f"feature count: {len(FEATURE_COLUMNS)}")

display(pd.DataFrame({"feature": FEATURE_COLUMNS}))
# display(train_fe[FEATURE_COLUMNS + [CONFIG.target_col]].head())

In [ ]:
# Block 4 - Model Training and Validation Scoring

# Build model input and remove rows without labels.
# Feature NaN is allowed because every MODEL_SPECS pipeline starts with SimpleImputer,
# but target NaN cannot be used by sklearn fit or metric functions.
def get_xy(df: pd.DataFrame, feature_columns: list[str], config: ExperimentConfig) -> tuple[pd.DataFrame, pd.Series]:
    x = df[feature_columns].replace([np.inf, -np.inf], np.nan).copy()
    y = pd.to_numeric(df[config.target_col], errors="coerce")

    valid_label_mask = y.notna()
    x = x.loc[valid_label_mask].copy()
    y = y.loc[valid_label_mask].copy()

    return x, y


# Expand the hyperparameter grid, but cap the number of candidates from CONFIG.
# This keeps notebook experiments reproducible and prevents accidental long searches.
def iter_param_candidates(param_grid: dict[str, list[Any]], config: ExperimentConfig) -> list[dict[str, Any]]:
    candidates = list(ParameterGrid(param_grid)) if param_grid else [{}]

    if len(candidates) <= config.max_param_candidates_per_model:
        return candidates

    rng = np.random.default_rng(config.random_seed)
    selected_idx = sorted(
        rng.choice(
            len(candidates),
            size=config.max_param_candidates_per_model,
            replace=False,
        ).tolist()
    )

    return [candidates[i] for i in selected_idx]


# Use TimeSeriesSplit instead of random CV to preserve temporal order.
# Random folds would leak future daily patterns into earlier validation folds.
def time_cv_score(
    estimator: Any,
    train_data: pd.DataFrame,
    feature_columns: list[str],
    params: dict[str, Any],
    config: ExperimentConfig,
) -> float:
    x_train, y_train = get_xy(train_data, feature_columns, config)

    n_splits = min(config.cv_splits, len(train_data) - 1)
    if n_splits < 2:
        return np.nan

    cv = TimeSeriesSplit(n_splits=n_splits)
    fold_scores = []

    for fold_train_idx, fold_valid_idx in cv.split(x_train):
        fold_estimator = clone(estimator)
        fold_estimator.set_params(**params)

        x_fold_train = x_train.iloc[fold_train_idx]
        y_fold_train = y_train.iloc[fold_train_idx]
        x_fold_valid = x_train.iloc[fold_valid_idx]
        y_fold_valid = y_train.iloc[fold_valid_idx]

        fold_estimator.fit(x_fold_train, y_fold_train)

        # Daily sales cannot be negative, so predictions are clipped at zero.
        fold_pred = fold_estimator.predict(x_fold_valid)
        fold_pred = np.maximum(fold_pred, 0)

        # Defensive metric mask: experimental feature expansion can produce
        # invalid predictions for some model/parameter combinations. Skip those
        # rows instead of letting one unstable candidate stop the full search.
        metric_mask = y_fold_valid.notna() & np.isfinite(fold_pred)
        if metric_mask.sum() == 0:
            fold_score = np.nan
        else:
            fold_score = mean_absolute_percentage_error(
                y_fold_valid.loc[metric_mask],
                fold_pred[metric_mask],
            ) * 100
        fold_scores.append(fold_score)

    return float(np.nanmean(fold_scores))


# Search the candidate parameters using only train data, then refit the selected
# configuration on the full train split.
def fit_model_with_search(
    model_name: str,
    model_spec: dict[str, Any],
    train_data: pd.DataFrame,
    feature_columns: list[str],
    config: ExperimentConfig,
) -> dict[str, Any]:
    estimator = model_spec["estimator"]
    param_grid = model_spec["param_grid"]

    best_params = None
    best_cv_mape = np.inf

    for params in iter_param_candidates(param_grid, config):
        cv_mape = time_cv_score(
            estimator=estimator,
            train_data=train_data,
            feature_columns=feature_columns,
            params=params,
            config=config,
        )

        if cv_mape < best_cv_mape:
            best_cv_mape = cv_mape
            best_params = params

    fitted_model = clone(estimator)
    fitted_model.set_params(**best_params)

    x_train, y_train = get_xy(train_data, feature_columns, config)
    fitted_model.fit(x_train, y_train)

    return {
        "model_name": model_name,
        "model": fitted_model,
        "best_params": best_params,
        "cv_daily_mape": best_cv_mape,
    }


# Generate row-level daily predictions. Month-level evaluation is intentionally
# deferred to the next block because the business metric is anchored monthly total.
def predict_daily_qty(
    model: Any,
    df: pd.DataFrame,
    feature_columns: list[str],
    config: ExperimentConfig,
) -> pd.DataFrame:
    x = df[feature_columns].copy()

    # Enforce the non-negative domain of daily sales.
    pred = model.predict(x)
    pred = np.maximum(pred, 0)

    out = df[
        [
            "split",
            "bizym",
            "anchor_date",
            "target_date",
            "horizon_days",
            config.target_col,
        ]
    ].copy()

    out["predicted_daily_qty"] = pred
    out["daily_error"] = out["predicted_daily_qty"] - out[config.target_col]
    out["daily_abs_error"] = out["daily_error"].abs()

    # Keep zero-label days as NaN APE to avoid infinite percentage errors.
    out["daily_ape"] = out["daily_abs_error"] / out[config.target_col].replace(0, np.nan) * 100

    return out


fitted_records = []

# Train each configured model family and keep all fitted models in memory.
# No artifacts are written because this notebook is a pure experiment.
for model_name, model_spec in MODEL_SPECS.items():
    fitted_record = fit_model_with_search(
        model_name=model_name,
        model_spec=model_spec,
        train_data=train_fe,
        feature_columns=FEATURE_COLUMNS,
        config=CONFIG,
    )

    split_predictions = []

    # Score all splits with the same fitted model so later blocks can evaluate
    # train/valid/test under the same monthly anchor rule.
    for split_df in [train_fe, valid_fe, test_fe]:
        split_pred = predict_daily_qty(
            model=fitted_record["model"],
            df=split_df,
            feature_columns=FEATURE_COLUMNS,
            config=CONFIG,
        )
        split_predictions.append(split_pred)

    fitted_record["daily_pred_df"] = pd.concat(split_predictions, ignore_index=True)
    fitted_records.append(fitted_record)


# Compact training summary: CV daily MAPE is only a tuning signal.
# Final model selection will use anchored monthly evaluation in the next blocks.
model_train_summary = pd.DataFrame(
    [
        {
            "model_name": record["model_name"],
            "cv_daily_mape": record["cv_daily_mape"],
            "best_params": record["best_params"],
        }
        for record in fitted_records
    ]
).sort_values("cv_daily_mape")

display(model_train_summary)